# Inductive logic programming — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def sigmoid(z): return 1/(1+np.exp(-z))
def bce(p,y):
    p=np.clip(np.asarray(p,dtype=float),1e-9,1-1e-9)
    y=np.asarray(y,dtype=float)
    return -(y*np.log(p)+(1-y)*np.log(1-p))
print('setup ok')

## Searching symbolic rules by coverage of examples
Inductive logic programming learns a logic rule from relational examples. These five examples keep the universe tiny and show how candidate clauses are scored by positive coverage, negative coverage, compression, and version-space shrinkage until the grandparent rule stands out.

In [ ]:
# 1) facts and examples: grandparent is a two-hop parent relation
people=['ann','bob','cara','dan']; parent={('ann','bob'),('bob','cara'),('cara','dan')}; positives={('ann','cara'),('bob','dan')}; negatives={('ann','dan'),('bob','cara'),('cara','ann')}
def parent2(x,z): return any((x,y) in parent and (y,z) in parent for y in people)
def parent1(x,z): return (x,z) in parent
pairs=list(itertools.product(people,people)); pred2={p for p in pairs if parent2(*p)}
plt.figure(figsize=(4,4)); M=np.array([[parent2(a,b) for b in people] for a in people],dtype=int); plt.imshow(M,cmap='Greens'); plt.xticks(range(4),people); plt.yticks(range(4),people); plt.title('two-hop parent candidates')
assert pred2==positives

In [ ]:
# 2) compare candidate rules by positive and negative coverage
cands={'parent(X,Z)':parent1,'parent2(X,Z)':parent2,'anything(X,Z)':lambda x,z: x!=z}
pos_cov=[]; neg_cov=[]
for f in cands.values():
    pos_cov.append(sum(f(*p) for p in positives)); neg_cov.append(sum(f(*n) for n in negatives))
plt.figure(figsize=(6,3)); x=np.arange(len(cands)); plt.bar(x-0.15,pos_cov,width=0.3,label='positive covered'); plt.bar(x+0.15,neg_cov,width=0.3,label='negative covered'); plt.xticks(x,list(cands.keys()),rotation=15); plt.legend(); plt.title('coverage separates useful from overbroad rules')
assert pos_cov==[0,2,2] and neg_cov==[1,0,3]

In [ ]:
# 3) compression rewards covering positives without covering negatives
scores=np.array(pos_cov)-np.array(neg_cov); names=list(cands.keys())
plt.figure(figsize=(6,3)); plt.bar(names,scores,color=['tab:gray','tab:green','tab:red']); plt.xticks(rotation=15); plt.ylabel('pos - neg'); plt.title('simple ILP score chooses parent2')
assert names[int(np.argmax(scores))]=='parent2(X,Z)' and int(scores.max())==2

In [ ]:
# 4) adding examples shrinks the version space of consistent rules
prefix_pos=[{('ann','cara')}, positives]; prefix_neg=[set(), {('ann','dan')}, negatives]
counts=[]; labels=[]
for ps in prefix_pos:
    for ns in prefix_neg:
        ok=sum(all(f(*p) for p in ps) and not any(f(*n) for n in ns) for f in cands.values())
        counts.append(ok); labels.append(f'+{len(ps)}, -{len(ns)}')
plt.figure(figsize=(6,3)); plt.plot(range(len(counts)),counts,'-o'); plt.xticks(range(len(counts)),labels,rotation=25); plt.ylabel('consistent candidates'); plt.title('examples prune inconsistent rules')
assert counts[-1]==1

In [ ]:
# 5) the learned rule generalizes to an unseen pair after adding one new fact
parent2_before=parent2('ann','dan'); parent.add(('bob','dan'))
parent2_after=parent2('ann','dan')
plt.figure(figsize=(6,3)); plt.bar(['before new fact','after new fact'],[parent2_before,parent2_after],color=['tab:gray','tab:green']); plt.ylim(0,1.1); plt.title('symbolic rule reuses new facts immediately')
assert parent2_before==False and parent2_after==True